In [1]:
import tensorflow as tf
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K
import numpy as np
import keras
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Model
from tensorflow.keras.utils import register_keras_serializable
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from glob import glob
import rasterio
import pandas as pd
import geopandas as gpd
from rasterio.transform import from_origin


## fOR utcnn

In [13]:
df_points = pd.read_csv("points.csv")
path_unc = 'unceratainty_final_model_outputs%.csv'
unc_A = pd.read_csv(path_unc)

In [14]:
unc_A[['X','Y']] = df_points[['X','Y']]
unc_A.to_csv(path_unc)
unc_A

,Unnamed: 0,mean_unc,std_unc,X,Y
0,0,61.402280,3.566416,3.343779,18.967783
1,1,62.463993,3.322849,3.352772,18.967783
2,2,60.156994,3.382251,3.361765,18.967783
3,3,60.513443,3.925512,3.370759,18.967783
4,4,60.716522,3.431363,3.379752,18.967783
...,...,...,...,...,...
2823295,2823295,-137.279430,115.584366,8.937562,37.538783
2823296,2823296,-48.344550,81.355630,8.946555,37.538783
2823297,2823297,31.248812,2.141286,8.946555,37.547776
2823298,2823298,30.642258,1.556897,8.946555,37.556769


In [ ]:
df=  unc_A
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.X, df.Y), crs="EPSG:4326")
gdf = gdf.to_crs(epsg=32631)  # UTM zone 31N

In [16]:
df['X_m'] = gdf.geometry.x
df['Y_m'] = gdf.geometry.y

In [17]:
# Use projected coordinates (UTM)
x = df['X_m'].values
y = df['Y_m'].values

variables=['mean_unc', 'std_unc']
property = 'S%'
for prop in variables: 
    values = df[prop].values

    pixel_size = 1000  # meters

    # Grid limits
    xmin, xmax = x.min(), x.max()
    ymin, ymax = y.min(), y.max()

    # Grid size
    nx = int(np.ceil((xmax - xmin) / pixel_size))
    ny = int(np.ceil((ymax - ymin) / pixel_size))

    # Convert points to grid indices
    ix = ((x - xmin) / pixel_size).astype(int)
    iy = ((ymax - y) / pixel_size).astype(int)  # flip y

    # Initialize raster with NaNs
    grid = np.full((ny, nx), np.nan, dtype=np.float32)

    # Assign values (nearest)
    grid[iy, ix] = values

    # Fill NaNs using nearest neighbor
    from scipy.ndimage import distance_transform_edt

    mask = np.isnan(grid)
    coords = np.array(np.nonzero(~mask)).T
    values_known = grid[~mask]

    # Compute distance transform to nearest known pixel
    filled_grid = distance_transform_edt(mask, return_distances=False, return_indices=True)
    grid_filled = grid[tuple(filled_grid)]

    # Save GeoTIFF
    transform = from_origin(xmin, ymax, pixel_size, pixel_size)

    with rasterio.open(
        f'./outputs_unceratinty_utcnn/{prop}_{property}_map.tif',
        'w',
        driver='GTiff',
        height=grid_filled.shape[0],
        width=grid_filled.shape[1],
        count=1,
        dtype=grid_filled.dtype,
        crs='EPSG:32631',
        transform=transform
    ) as dst:
        dst.write(grid_filled, 1)

# For MTCNN 

In [8]:
df_points = pd.read_csv("points.csv")
path_unc = 'uncertainty_final_model_outputs%_fortasks_new.csv'
unc_A = pd.read_csv(path_unc)
unc_A[['X','Y']] = df_points[['X','Y']]
unc_A.to_csv(path_unc)
unc_A

,Q05_task0,Q50_task0,Q95_task0,rel_unc_task0,rel_unc_task0_int,Q05_task1,Q50_task1,Q95_task1,rel_unc_task1,rel_unc_task1_int,...,Q95_task4,rel_unc_task4,rel_unc_task4_int,Q05_task5,Q50_task5,Q95_task5,rel_unc_task5,rel_unc_task5_int,X,Y
0,18.316994,26.991680,44.280655,0.961913,9,29.284950,35.500156,49.190440,0.560716,5,...,59.892982,0.410793,4,42.706960,54.970726,65.995125,0.423647,4,3.343779,18.967783
1,20.495348,30.670021,45.854458,0.826837,8,32.994335,40.070100,47.892624,0.371806,3,...,63.525505,0.342565,3,49.352030,58.566970,67.420390,0.308508,3,3.352772,18.967783
2,15.977318,24.910955,31.559437,0.625513,6,26.144552,32.213516,40.890800,0.457766,4,...,57.668465,0.414821,4,43.060658,53.322968,62.801598,0.370215,3,3.361765,18.967783
3,14.260408,24.645859,36.351067,0.896323,8,26.442880,31.526104,41.527660,0.478485,4,...,53.508293,0.344178,3,40.741764,49.225098,60.104770,0.393356,3,3.370759,18.967783
4,15.885543,22.566757,32.202050,0.723033,7,25.307507,28.986294,37.893505,0.434205,4,...,50.829483,0.427493,4,34.667618,43.469910,51.091652,0.377825,3,3.379752,18.967783
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2823295,19.298037,29.294320,41.355354,0.752955,7,58.535477,80.427925,88.260820,0.369590,3,...,52.884274,0.931638,9,37.099995,45.883610,61.839165,0.539172,5,8.937562,37.538783
2823296,17.190014,26.597164,35.465600,0.687125,6,56.772500,81.711426,91.446434,0.424346,4,...,48.209244,1.204906,12,35.127644,42.602547,57.477478,0.524613,5,8.946555,37.538783
2823297,15.809718,24.199806,33.416664,0.727565,7,39.727390,76.468580,89.582080,0.651963,6,...,40.555000,1.288737,12,33.846416,41.615730,55.649070,0.523904,5,8.946555,37.547776
2823298,11.951460,20.675041,29.921726,0.869177,8,32.703316,61.342660,84.709330,0.847795,8,...,41.504160,0.802021,8,19.828703,28.489712,35.955110,0.566043,5,8.946555,37.556769


In [9]:
df=  unc_A
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.X, df.Y), crs="EPSG:4326")
gdf = gdf.to_crs(epsg=32631)  # UTM zone 31N

In [10]:
df['X_m'] = gdf.geometry.x
df['Y_m'] = gdf.geometry.y

In [11]:
depth_cols = [
    'unc_task0_int',
    'unc_task1_int',
    'unc_task2_int',
    'unc_task3_int',
    'unc_task4_int',
    'unc_task5_int'
]

In [12]:
# Use projected coordinates (UTM)
x = df['X_m'].values
y = df['Y_m'].values

variables=['rel']
property = 'sand%'
for prop in variables: 

    for col in depth_cols:

        values = df[f'{prop}_{col}'].values

        pixel_size = 1000  # meters

        # Grid limits
        xmin, xmax = x.min(), x.max()
        ymin, ymax = y.min(), y.max()

        # Grid size
        nx = int(np.ceil((xmax - xmin) / pixel_size))
        ny = int(np.ceil((ymax - ymin) / pixel_size))

        # Convert points to grid indices
        ix = ((x - xmin) / pixel_size).astype(int)
        iy = ((ymax - y) / pixel_size).astype(int)  # flip y

        # Initialize raster with NaNs
        grid = np.full((ny, nx), np.nan, dtype=np.float32)

        # Assign values (nearest)
        grid[iy, ix] = values

        # Fill NaNs using nearest neighbor
        from scipy.ndimage import distance_transform_edt

        mask = np.isnan(grid)
        coords = np.array(np.nonzero(~mask)).T
        values_known = grid[~mask]

        # Compute distance transform to nearest known pixel
        filled_grid = distance_transform_edt(mask, return_distances=False, return_indices=True)
        grid_filled = grid[tuple(filled_grid)]

        # Save GeoTIFF
        transform = from_origin(xmin, ymax, pixel_size, pixel_size)

        with rasterio.open(
            f'./outputs_unceratinty_mtcnn/{prop}_depth_{col}_{property}_map_new.tif',
            'w',
            driver='GTiff',
            height=grid_filled.shape[0],
            width=grid_filled.shape[1],
            count=1,
            dtype=grid_filled.dtype,
            crs='EPSG:32631',
            transform=transform
        ) as dst:
            dst.write(grid_filled, 1)

## remove outliers

In [19]:
import rasterio
import numpy as np
from pathlib import Path

input_dir = Path("outputs_unceratinty_utcnn")
output_dir = Path("sand_uncer_without_outliers")
output_dir.mkdir(exist_ok=True)

for tif_path in input_dir.glob("*S%_map.tif"):
    with rasterio.open(tif_path) as src:
        data = src.read(1).astype(np.float32)
        profile = src.profile

    # Mask values > 90
    valid_mask = data <= 90
    valid_data = data[valid_mask]

    if valid_data.size == 0:
        print(f"Skipping {tif_path.name}: no valid data")
        continue

    # Compute 95th percentile
    p95 = np.percentile(valid_data, 95)

    # Cap values above p95
    data_clipped = np.where(data > p95, p95, data)

    # Save new TIFF
    profile.update(dtype=rasterio.float32)

    out_path = output_dir / tif_path.name
    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(data_clipped, 1)

    print(f"Processed: {tif_path.name} | p95 = {p95:.2f}")


Processed: mean_unc_S%_map.tif | p95 = 86.17
Processed: std_unc_S%_map.tif | p95 = 4.69


## For MLP 

In [13]:
df_points = pd.read_csv("points.csv")
path_unc = 'outputs_uncertainty_mlp_np\\uncertainties_mlp.csv'
unc_A = pd.read_csv(path_unc)
unc_A[['X','Y']] = df_points[['X','Y']]
unc_A.to_csv(path_unc)
unc_A

,Unnamed: 0,mean_unc_pH_values,std_unc_pH_values,Q05_prop_pH,Q50_prop_pH,Q95_prop_pH,rel_unc_prop_pH,rel_unc_prop_pH_int,mean_unc_CE_values,std_unc_CE_values,...,rel_unc_prop_L%_int,mean_unc_S%_values,std_unc_S%_values,Q05_prop_S%,Q50_prop_S%,Q95_prop_S%,rel_unc_prop_S%,rel_unc_prop_S%_int,X,Y
0,0,7.748385,0.166554,7.506917,7.730101,8.026474,0.067212,0,0.817680,1.975758,...,29,104.668150,9.161018,89.895330,106.502100,117.586105,0.260002,2,3.343779,18.967783
1,1,7.789629,0.179264,7.516610,7.773142,8.057065,0.069528,0,1.172464,1.818861,...,13,96.027800,6.598682,85.103836,95.535900,107.224500,0.231543,2,3.352772,18.967783
2,2,7.752939,0.166782,7.519341,7.737629,8.047167,0.068215,0,0.638347,1.703378,...,15,96.952750,6.734650,86.354645,97.314255,108.711860,0.229742,2,3.361765,18.967783
3,3,7.761490,0.166696,7.509265,7.746730,8.028450,0.067020,0,1.465521,1.727651,...,13,94.885260,6.935288,85.820400,94.625260,107.279140,0.226776,2,3.370759,18.967783
4,4,7.781602,0.160826,7.586155,7.746771,8.090287,0.065076,0,1.420006,1.764084,...,12,93.728050,6.513250,83.406395,94.075910,102.744950,0.205563,2,3.379752,18.967783
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2823295,2823295,7.977803,0.113572,7.797493,7.978914,8.180053,0.047946,0,14.332890,3.520026,...,7,45.430473,15.667971,20.891315,44.198900,70.366035,1.119366,11,8.937562,37.538783
2823296,2823296,7.465208,0.063807,7.362816,7.468686,7.564462,0.026999,0,22.739988,1.323085,...,0,34.373512,1.562893,31.993027,34.340416,36.773770,0.139216,1,8.946555,37.538783
2823297,2823297,7.391954,0.105146,7.221025,7.399510,7.541086,0.043254,0,16.854765,2.104627,...,1,46.161186,4.399797,34.444260,47.021034,51.468628,0.362059,3,8.946555,37.547776
2823298,2823298,7.665681,0.033264,7.670339,7.670339,7.670339,0.000000,0,14.701940,1.150673,...,0,42.367600,1.145904,42.530420,42.530420,42.530420,0.000000,0,8.946555,37.556769


In [14]:
df=  unc_A
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.X, df.Y), crs="EPSG:4326")
gdf = gdf.to_crs(epsg=32631)  # UTM zone 31N

In [15]:
df.columns

Index(['Unnamed: 0', 'mean_unc_pH_values', 'std_unc_pH_values', 'Q05_prop_pH',
       'Q50_prop_pH', 'Q95_prop_pH', 'rel_unc_prop_pH', 'rel_unc_prop_pH_int',
       'mean_unc_CE_values', 'std_unc_CE_values', 'Q05_prop_CE', 'Q50_prop_CE',
       'Q95_prop_CE', 'rel_unc_prop_CE', 'rel_unc_prop_CE_int',
       'mean_unc_CEC_values', 'std_unc_CEC_values', 'Q05_prop_CEC',
       'Q50_prop_CEC', 'Q95_prop_CEC', 'rel_unc_prop_CEC',
       'rel_unc_prop_CEC_int', 'mean_unc_OM_values', 'std_unc_OM_values',
       'Q05_prop_OM', 'Q50_prop_OM', 'Q95_prop_OM', 'rel_unc_prop_OM',
       'rel_unc_prop_OM_int', 'mean_unc_A%_values', 'std_unc_A%_values',
       'Q05_prop_A%', 'Q50_prop_A%', 'Q95_prop_A%', 'rel_unc_prop_A%',
       'rel_unc_prop_A%_int', 'mean_unc_L%_values', 'std_unc_L%_values',
       'Q05_prop_L%', 'Q50_prop_L%', 'Q95_prop_L%', 'rel_unc_prop_L%',
       'rel_unc_prop_L%_int', 'mean_unc_S%_values', 'std_unc_S%_values',
       'Q05_prop_S%', 'Q50_prop_S%', 'Q95_prop_S%', 'rel_unc_prop

In [20]:
import re

props = ['pH', 'CE', 'CEC', 'OM', 'A%', 'L%', 'S%']
props_pattern = '|'.join(map(re.escape, props))

def rename_col(col):
    if col.endswith("_values"):
        col = col[: -len("_values")]
    # Move _int before property name if it appears at the end
    col = re.sub(rf'(.*)_({props_pattern})_int$', r'\1_int_\2', col)
    return col

df.columns = [rename_col(c) for c in df.columns]

df.columns

Index(['Unnamed: 0', 'mean_unc_pH', 'std_unc_pH', 'Q05_prop_pH', 'Q50_prop_pH',
       'Q95_prop_pH', 'rel_unc_prop_pH', 'rel_unc_prop_int_pH', 'mean_unc_CE',
       'std_unc_CE', 'Q05_prop_CE', 'Q50_prop_CE', 'Q95_prop_CE',
       'rel_unc_prop_CE', 'rel_unc_prop_int_CE', 'mean_unc_CEC', 'std_unc_CEC',
       'Q05_prop_CEC', 'Q50_prop_CEC', 'Q95_prop_CEC', 'rel_unc_prop_CEC',
       'rel_unc_prop_int_CEC', 'mean_unc_OM', 'std_unc_OM', 'Q05_prop_OM',
       'Q50_prop_OM', 'Q95_prop_OM', 'rel_unc_prop_OM', 'rel_unc_prop_int_OM',
       'mean_unc_A%', 'std_unc_A%', 'Q05_prop_A%', 'Q50_prop_A%',
       'Q95_prop_A%', 'rel_unc_prop_A%', 'rel_unc_prop_int_A%', 'mean_unc_L%',
       'std_unc_L%', 'Q05_prop_L%', 'Q50_prop_L%', 'Q95_prop_L%',
       'rel_unc_prop_L%', 'rel_unc_prop_int_L%', 'mean_unc_S%', 'std_unc_S%',
       'Q05_prop_S%', 'Q50_prop_S%', 'Q95_prop_S%', 'rel_unc_prop_S%',
       'rel_unc_prop_int_S%', 'X', 'Y', 'X_m', 'Y_m'],
      dtype='object')

In [21]:
df['X_m'] = gdf.geometry.x
df['Y_m'] = gdf.geometry.y

In [22]:
cols = [
    'mean_unc',
    'std_unc',
    'rel_unc_prop_int'
]

In [25]:
# Use projected coordinates (UTM)
x = df['X_m'].values
y = df['Y_m'].values

variables= ['pH', 'CE', 'CEC', 'OM' ,'A%', 'L%', 'S%']

for prop in variables: 

    for col in cols:

        values = df[f'{col}_{prop}'].values

        pixel_size = 1000  # meters

        # Grid limits
        xmin, xmax = x.min(), x.max()
        ymin, ymax = y.min(), y.max()

        # Grid size
        nx = int(np.ceil((xmax - xmin) / pixel_size))
        ny = int(np.ceil((ymax - ymin) / pixel_size))

        # Convert points to grid indices
        ix = ((x - xmin) / pixel_size).astype(int)
        iy = ((ymax - y) / pixel_size).astype(int)  # flip y

        # Initialize raster with NaNs
        grid = np.full((ny, nx), np.nan, dtype=np.float32)

        # Assign values (nearest)
        grid[iy, ix] = values

        # Fill NaNs using nearest neighbor
        from scipy.ndimage import distance_transform_edt

        mask = np.isnan(grid)
        coords = np.array(np.nonzero(~mask)).T
        values_known = grid[~mask]

        # Compute distance transform to nearest known pixel
        filled_grid = distance_transform_edt(mask, return_distances=False, return_indices=True)
        grid_filled = grid[tuple(filled_grid)]

        # Save GeoTIFF
        transform = from_origin(xmin, ymax, pixel_size, pixel_size)

        with rasterio.open(
            f'./outputs_uncertainty_mlp/{prop}_{col}_map_new.tif',
            'w',
            driver='GTiff',
            height=grid_filled.shape[0],
            width=grid_filled.shape[1],
            count=1,
            dtype=grid_filled.dtype,
            crs='EPSG:32631',
            transform=transform
        ) as dst:
            dst.write(grid_filled, 1)

# Calculate the PICP ...

In [2]:
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
import numpy as np
import pandas as pd
import rasterio


In [3]:
Z_SCORES = {
    0.68: 1.0,
    0.90: 1.645,
    0.95: 1.96
}

def compute_picp(y_true, mean, std, confidence=0.95):
    z = Z_SCORES[confidence]
    lower = mean - z * std
    upper = mean + z * std
    return np.mean((y_true >= lower) & (y_true <= upper))

def compute_mpiw(std, confidence=0.95):
    z = Z_SCORES[confidence]
    return np.mean(2 * z * std)

def compute_piw(std, confidence=0.95):
    z = Z_SCORES[confidence]
    return 2 * z * std


In [4]:
# def load_and_merge(
#     uncertainty_csv,
#     observations_csv,
#     property_name
# ):
#     """
#     Merge uncertainty CSV with observed values for one property
#     """

#     df_u = pd.read_csv(uncertainty_csv)
#     print(df_u.head())
#     df_obs = pd.read_csv(observations_csv)
#     print(df_obs.head())

#     df = pd.merge(
#         df_u,
#         df_obs[["X", "Y", property_name]],
#         on=["X", "Y"],
#         how="inner"
#     )

#     df = df.rename(columns={property_name: "obs"})

#     return df

def spatial_merge(
    df_pred,
    df_obs,
    x_col="X",
    y_col="Y",
    obs_value_col=None,
    max_distance=5.0
):
    """
    Spatially merge prediction & observation points using nearest neighbor

    max_distance in CRS units (meters if projected CRS)
    """

    # Build KDTree on observed points
    tree = cKDTree(df_obs[[x_col, y_col]].values)

    # Query nearest observed point for each prediction
    dist, idx = tree.query(
        df_pred[[x_col, y_col]].values,
        distance_upper_bound=max_distance
    )

    # Valid matches
    valid = dist != np.inf

    merged = df_pred.loc[valid].copy()
    matched_obs = df_obs.iloc[idx[valid]].copy()

    merged["obs"] = matched_obs[obs_value_col].values
    merged["match_dist"] = dist[valid]

    return merged

def spatial_merge_obs_to_pred(
    df_obs,
    df_pred,
    obs_value_col,
    x_col="X",
    y_col="Y",
    max_distance=5.0
):
    """
    For each observation point, find nearest prediction (mean, std).
    This behaves like an INNER JOIN on observations.
    """

    # Build KDTree on prediction points (many)
    tree = cKDTree(df_pred[[x_col, y_col]].values)

    # Query nearest prediction for each observation
    dist, idx = tree.query(
        df_obs[[x_col, y_col]].values,
        distance_upper_bound=max_distance
    )

    # Keep only valid matches
    valid = dist != np.inf

    merged = df_obs.loc[valid].copy()
    matched_pred = df_pred.iloc[idx[valid]].copy()

    merged["mean_unc"] = matched_pred["mean_unc"].values
    merged["std_unc"] = matched_pred["std_unc"].values
    merged["match_dist"] = dist[valid]

    merged = merged.rename(columns={obs_value_col: "obs"})

    return merged

def load_and_merge(
    uncertainty_csv,
    observations_csv,
    property_name,
    max_distance=1.0
):
    df_pred = pd.read_csv(uncertainty_csv)
    df_obs = pd.read_csv(observations_csv)

    df = spatial_merge_obs_to_pred(
        df_obs=df_obs,
        df_pred=df_pred,
        obs_value_col=property_name,
        max_distance=max_distance
    )

    return df


In [5]:
def evaluate_property(
    uncertainty_csv,
    observations_csv,
    property_name,
    confidence_levels=(0.68, 0.90, 0.95)
):
    df = load_and_merge(
        uncertainty_csv,
        observations_csv,
        property_name
    )

    print(f"\n📊 Evaluation for {property_name}")
    print(f"Number of validation points: {len(df)}")

    results = []

    for c in confidence_levels:
        z = Z_SCORES[c]

        # ---- Prediction interval bounds
        lower = df["mean_unc"] - z * df["std_unc"]
        upper = df["mean_unc"] + z * df["std_unc"]

        # ---- Coverage
        covered = (df["obs"] >= lower) & (df["obs"] <= upper)
        picp = covered.mean()

        # ---- PIW per point
        piw = 2 * z * df["std_unc"]

        # ---- MPIW (mean PIW)
        mpiw = piw.mean()

        # ---- Optional robust stats
        piw_median = piw.median()
        piw_q25 = piw.quantile(0.25)
        piw_q75 = piw.quantile(0.75)

        results.append({
            "property": property_name,
            "confidence": c,
            "PICP": picp,
            "MPIW": mpiw,
            "PIW_median": piw_median,
            "PIW_q25": piw_q25,
            "PIW_q75": piw_q75
        })

        print(
            f"  {int(c*100)}% → "
            f"PICP = {picp:.3f}, "
            f"MPIW = {mpiw:.3f}, "
            f"PIW(median) = {piw_median:.3f}"
        )

    return pd.DataFrame(results)


In [22]:
all_results = []

# A
all_results.append(
    evaluate_property(
        "unceratainty_A.csv",
        "../data/dataset.csv",
        "A%"
    )
)

# L
all_results.append(
    evaluate_property(
        "unceratainty_final_model_outputl.csv",
        "../data/dataset.csv",
        "L%"
    )
)

# L
all_results.append(
    evaluate_property(
        "unceratainty_final_model_outputs%.csv",
        "../data/dataset.csv",
        "S%"
    )
)
# ce
all_results.append(
    evaluate_property(
        "unceratainty_final_model_outputce.csv",
        "../data/dataset.csv",
        "CE"
    )
)

# cec
all_results.append(
    evaluate_property(
        "unceratainty_final_model_outputcec.csv",
        "../data/dataset.csv",
        "CEC"
    )
)

# cec
all_results.append(
    evaluate_property(
        "unceratainty_final_model_outputph.csv",
        "../data/dataset.csv",
        "pH"
    )
)

# SOM
all_results.append(
    evaluate_property(
        "unceratainty_final_model_outputom(%).csv",
        "../data/dataset.csv",
        "OM"
    )
)



📊 Evaluation for A%
Number of validation points: 2156
  68% → PICP = 0.416, MPIW = 6.898, PIW(median) = 6.795
  90% → PICP = 0.559, MPIW = 11.348, PIW(median) = 11.178
  95% → PICP = 0.615, MPIW = 13.521, PIW(median) = 13.319

📊 Evaluation for L%
Number of validation points: 2156
  68% → PICP = 0.203, MPIW = 4.135, PIW(median) = 3.388
  90% → PICP = 0.343, MPIW = 6.802, PIW(median) = 5.573
  95% → PICP = 0.416, MPIW = 8.105, PIW(median) = 6.640

📊 Evaluation for S%
Number of validation points: 2156
  68% → PICP = 0.154, MPIW = 5.430, PIW(median) = 4.981
  90% → PICP = 0.239, MPIW = 8.932, PIW(median) = 8.194
  95% → PICP = 0.293, MPIW = 10.642, PIW(median) = 9.763

📊 Evaluation for CE
Number of validation points: 2156
  68% → PICP = 0.119, MPIW = 1.041, PIW(median) = 0.964
  90% → PICP = 0.185, MPIW = 1.712, PIW(median) = 1.586
  95% → PICP = 0.212, MPIW = 2.040, PIW(median) = 1.889

📊 Evaluation for CEC
Number of validation points: 2156
  68% → PICP = 0.136, MPIW = 1.123, PIW(median)

In [23]:
results_df = pd.concat(all_results, ignore_index=True)

results_df.to_csv(
    "uncertainty_evaluation_picp_mpiw_piw_cktree.csv",
    index=False
)

print("\n✅ Results saved to uncertainty_evaluation_picp_mpiw.csv")



✅ Results saved to uncertainty_evaluation_picp_mpiw.csv


In [6]:
def sample_raster_at_points(src, xs, ys):
    rows, cols = src.index(xs, ys)
    return src.read(1)[rows, cols]

# def sample_prediction_rasters(
#     mean_tif,
#     std_tif,
#     obs_csv,
#     property_name,
#     x_col="X",
#     y_col="Y"
# ):
#     """
#     Sample mean and std rasters at observation point locations
#     """

#     df_obs = pd.read_csv(obs_csv)

#     coords = list(zip(df_obs[x_col], df_obs[y_col]))

#     with rasterio.open(mean_tif) as src_mean, rasterio.open(std_tif) as src_std:
#         # mean_vals = np.array([v[0] for v in src_mean.sample(coords)])
#         # std_vals = np.array([v[0] for v in src_std.sample(coords)])
#         mean_vals = sample_raster_at_points(
#         src_mean,
#         df_obs["X"].values,
#         df_obs["Y"].values
#         )
#         std_vals = sample_raster_at_points(
#         src_std,
#         df_obs["X"].values,
#         df_obs["Y"].values
#         )
#         print(mean_vals,std_vals)
#         nodata_mean = src_mean.nodata
#         nodata_std = src_std.nodata

#     df_obs["mean"] = mean_vals
#     df_obs["std"] = std_vals

#     # Remove NoData points
#     if nodata_mean is not None:
#         df_obs = df_obs[df_obs["mean"] != nodata_mean]

#     if nodata_std is not None:
#         df_obs = df_obs[df_obs["std"] != nodata_std]

#     df_obs = df_obs.rename(columns={property_name: "obs"})

#     return df_obs[["X", "Y", "obs", "mean", "std"]]

def sample_prediction_rasters(
    mean_tif,
    std_tif,
    obs_csv,
    property_name
):
    df_obs = pd.read_csv(obs_csv)
    with rasterio.open(mean_tif) as src:
        raster_crs = src.crs
        bounds = src.bounds

    # Reproject if needed
    if df_obs.attrs.get("crs") != raster_crs:
        from pyproj import Transformer
        transformer = Transformer.from_crs(
            "EPSG:4326", raster_crs, always_xy=True
        )
        df_obs["X"], df_obs["Y"] = transformer.transform(
            df_obs["X"].values,
            df_obs["Y"].values
        )

    coords = list(zip(df_obs["X"], df_obs["Y"]))

    with rasterio.open(mean_tif) as src_mean, rasterio.open(std_tif) as src_std:
        mean_vals = np.ma.filled(
            [v[0] for v in src_mean.sample(coords, masked=True)],
            np.nan
        )
        std_vals = np.ma.filled(
            [v[0] for v in src_std.sample(coords, masked=True)],
            np.nan
        )

    df = df_obs.copy()
    df["mean"] = mean_vals
    df["std"] = std_vals
    df = df.rename(columns={property_name: "obs"})

    return df.dropna(subset=["mean", "std"])


In [7]:
def evaluate_property_from_rasters(
    mean_tif,
    std_tif,
    obs_csv,
    property_name,
    confidence_levels=(0.68, 0.90, 0.95)
):
    df = sample_prediction_rasters(
        mean_tif=mean_tif,
        std_tif=std_tif,
        obs_csv=obs_csv,
        property_name=property_name
    )

    print(f"\n📊 Evaluation for {property_name}")
    print(f"Number of validation points: {len(df)}")
    # print(df)
    results = []

    for c in confidence_levels:
        picp = compute_picp(
            y_true=df["obs"].values,
            mean=df["mean"].values,
            std=df["std"].values,
            confidence=c
        )

        mpiw = compute_mpiw(
            std=df["std"].values,
            confidence=c
        )

        results.append({
            "property": property_name,
            "confidence": c,
            "PICP": picp,
            "MPIW": mpiw
        })

        print(
            f"  {int(c*100)}% → "
            f"PICP = {picp:.3f}, "
            f"MPIW = {mpiw:.3f}"
        )

    return pd.DataFrame(results)


In [8]:
def evaluate_property_from_rasters(
    mean_tif,
    std_tif,
    obs_csv,
    property_name,
    confidence_levels=(0.68, 0.90, 0.95)
):
    df = sample_prediction_rasters(
        mean_tif=mean_tif,
        std_tif=std_tif,
        obs_csv=obs_csv,
        property_name=property_name
    )

    print(f"\n📊 Evaluation for {property_name}")
    print(f"Number of validation points: {len(df)}")

    results = []

    for c in confidence_levels:
        z = Z_SCORES[c]

        # --- Prediction interval bounds
        lower = df["mean"] - z * df["std"]
        upper = df["mean"] + z * df["std"]

        # --- Coverage (PICP)
        covered = (df["obs"] >= lower) & (df["obs"] <= upper)
        picp = covered.mean()

        # --- PIW per point
        piw = 2 * z * df["std"]

        # --- MPIW (mean PIW)
        mpiw = piw.mean()

        # --- Robust PIW statistics (recommended)
        piw_median = piw.median()
        piw_q25 = piw.quantile(0.25)
        piw_q75 = piw.quantile(0.75)

        results.append({
            "property": property_name,
            "confidence": c,
            "PICP": picp,
            "MPIW": mpiw,
            "PIW_median": piw_median,
            "PIW_q25": piw_q25,
            "PIW_q75": piw_q75
        })

        print(
            f"  {int(c*100)}% → "
            f"PICP = {picp:.3f}, "
            f"MPIW = {mpiw:.3f}, "
            f"PIW(median) = {piw_median:.3f}"
        )

    return pd.DataFrame(results)


In [27]:
all_results = []

all_results.append(
    evaluate_property_from_rasters(
        mean_tif="outputs_unceratinty_utcnn\mean_unc_A%_map.tif",
        std_tif="outputs_unceratinty_utcnn\std_unc_A%_map.tif",
        obs_csv="../data/dataset.csv",
        property_name="A%"
    )
)

all_results.append(
    evaluate_property_from_rasters(
        mean_tif="outputs_unceratinty_utcnn\mean_unc_L%_map.tif",
        std_tif="outputs_unceratinty_utcnn\std_unc_L%_map.tif",
        obs_csv="../data/dataset.csv",
        property_name="L%"
    )
)

all_results.append(
    evaluate_property_from_rasters(
        mean_tif="outputs_unceratinty_utcnn\mean_unc_S%_map.tif",
        std_tif="outputs_unceratinty_utcnn\std_unc_S%_map.tif",
        obs_csv="../data/dataset.csv",
        property_name="S%"
    )
)


all_results.append(
    evaluate_property_from_rasters(
        mean_tif="outputs_unceratinty_utcnn\mean_unc_ce_map.tif",
        std_tif="outputs_unceratinty_utcnn\std_unc_ce_map.tif",
        obs_csv="../data/dataset.csv",
        property_name="CE"
    )
)

all_results.append(
    evaluate_property_from_rasters(
        mean_tif="outputs_unceratinty_utcnn\mean_unc_cec_map.tif",
        std_tif="outputs_unceratinty_utcnn\std_unc_cec_map.tif",
        obs_csv="../data/dataset.csv",
        property_name="CEC"
    )
)

all_results.append(
    evaluate_property_from_rasters(
        mean_tif="outputs_unceratinty_utcnn\mean_unc_pH_map.tif",
        std_tif="outputs_unceratinty_utcnn\std_unc_pH_map.tif",
        obs_csv="../data/dataset.csv",
        property_name="pH"
    )
)

all_results.append(
    evaluate_property_from_rasters(
        mean_tif="outputs_unceratinty_utcnn\mean_unc_SOM(%)_map.tif",
        std_tif="outputs_unceratinty_utcnn\std_unc_SOM(%)_map.tif",
        obs_csv="../data/dataset.csv",
        property_name="OM"
    )
)


📊 Evaluation for A%
Number of validation points: 2156
  68% → PICP = 0.416, MPIW = 6.847, PIW(median) = 6.839
  90% → PICP = 0.556, MPIW = 11.263, PIW(median) = 11.251
  95% → PICP = 0.619, MPIW = 13.420, PIW(median) = 13.405

📊 Evaluation for L%
Number of validation points: 2156
  68% → PICP = 0.198, MPIW = 4.083, PIW(median) = 3.303
  90% → PICP = 0.322, MPIW = 6.716, PIW(median) = 5.434
  95% → PICP = 0.392, MPIW = 8.002, PIW(median) = 6.475

📊 Evaluation for S%
Number of validation points: 2156
  68% → PICP = 0.137, MPIW = 5.483, PIW(median) = 4.973
  90% → PICP = 0.237, MPIW = 9.020, PIW(median) = 8.181
  95% → PICP = 0.297, MPIW = 10.748, PIW(median) = 9.747

📊 Evaluation for CE
Number of validation points: 2156
  68% → PICP = 0.114, MPIW = 1.053, PIW(median) = 0.969
  90% → PICP = 0.176, MPIW = 1.731, PIW(median) = 1.593
  95% → PICP = 0.203, MPIW = 2.063, PIW(median) = 1.899

📊 Evaluation for CEC
Number of validation points: 2156
  68% → PICP = 0.141, MPIW = 1.132, PIW(median)

### using csv predictions with cktree

📊 Evaluation for A%
Number of validation points: 2156
  68% → PICP = 0.416, MPIW = 6.898
  90% → PICP = 0.559, MPIW = 11.348
  95% → PICP = 0.615, MPIW = 13.521

📊 Evaluation for CE
Number of validation points: 2156
  68% → PICP = 0.119, MPIW = 1.041
  90% → PICP = 0.185, MPIW = 1.712
  95% → PICP = 0.212, MPIW = 2.040

📊 Evaluation for CEC
Number of validation points: 2156
  68% → PICP = 0.136, MPIW = 1.123
  90% → PICP = 0.311, MPIW = 1.848
  95% → PICP = 0.404, MPIW = 2.202

### using tif predictions
📊 Evaluation for A%
Number of validation points: 2156
  68% → PICP = 0.292, MPIW = 6.847
  90% → PICP = 0.468, MPIW = 11.263
  95% → PICP = 0.535, MPIW = 13.420

📊 Evaluation for CE
Number of validation points: 2156
  68% → PICP = 0.122, MPIW = 1.053
  90% → PICP = 0.183, MPIW = 1.731
  95% → PICP = 0.214, MPIW = 2.063

📊 Evaluation for CEC
Number of validation points: 2156
  68% → PICP = 0.109, MPIW = 1.132
  90% → PICP = 0.236, MPIW = 1.862
  95% → PICP = 0.293, MPIW = 2.219

In [28]:
results_df = pd.concat(all_results, ignore_index=True)

results_df.to_csv(
    "uncertainty_evaluation_picp_mpiw_piw_rasters.csv",
    index=False
)

print("\n✅ Results saved to uncertainty_evaluation_picp_mpiw.csv")



✅ Results saved to uncertainty_evaluation_picp_mpiw.csv


## Calculations by region

In [9]:
import pandas as pd
def split_north_south_algeria(
    obs_csv,
    lat_col="Y",
    lon_col="X",
    split_lat=34.0
):
    """
    Split observation points into North and South Algeria
    using latitude threshold (EPSG:4326)
    """

    df = pd.read_csv(obs_csv)

    north = df[df[lat_col] >= split_lat].copy()
    south = df[df[lat_col] < split_lat].copy()

    print(f"North Algeria points: {len(north)}")
    print(f"South Algeria points: {len(south)}")

    return north, south


In [10]:
from pyproj import Transformer
import rasterio

def reproject_df_to_raster(df, raster_tif, x_col="X", y_col="Y"):
    with rasterio.open(raster_tif) as src:
        dst_crs = src.crs

    transformer = Transformer.from_crs(
        "EPSG:4326", dst_crs, always_xy=True
    )

    df = df.copy()
    df[x_col], df[y_col] = transformer.transform(
        df[x_col].values,
        df[y_col].values
    )

    return df

import numpy as np
def sample_rasters_at_points(mean_tif, std_tif, df, property_name):
    coords = list(zip(df["X"], df["Y"]))

    with rasterio.open(mean_tif) as src_mean, rasterio.open(std_tif) as src_std:
        mean_vals = np.ma.filled(
            [v[0] for v in src_mean.sample(coords, masked=True)],
            np.nan
        )
        std_vals = np.ma.filled(
            [v[0] for v in src_std.sample(coords, masked=True)],
            np.nan
        )
    out = df.copy()
    out["mean"] = mean_vals
    out["std"] = std_vals
    out = out.rename(columns={property_name: "obs"})

    return out.dropna(subset=["mean", "std"])

def evaluate_region(
    df_obs,
    mean_tif,
    std_tif,
    property_name,
    region_name,
    confidence_levels=(0.68, 0.90, 0.95)
):
    df_obs_proj = reproject_df_to_raster(df_obs, mean_tif)

    df = sample_rasters_at_points(
        mean_tif,
        std_tif,
        df_obs_proj,
        property_name
    )

    print(f"\n📍 {region_name} — {property_name}")
    print(f"Validation points: {len(df)}")

    results = []

    for c in confidence_levels:
        picp = compute_picp(df["obs"], df["mean"], df["std"], c)
        mpiw = compute_mpiw(df["std"], c)

        print(f"  {int(c*100)}% → PICP={picp:.3f}, MPIW={mpiw:.3f}")

        results.append({
            "property": property_name,
            "region": region_name,
            "confidence": c,
            "PICP": picp,
            "MPIW": mpiw,
            "n_points": len(df)
        })

    return pd.DataFrame(results)


In [11]:
obs_csv="../data/dataset.csv"

north_df, south_df = split_north_south_algeria(obs_csv)
results = []

for prop in ["A%","L%", 'S%', "CE", "CEC", "pH" , "OM"]:
    results.append(
        evaluate_region(
            north_df,
            mean_tif=f"outputs_unceratinty_utcnn/mean_unc_{prop}_map.tif",
            std_tif=f"outputs_unceratinty_utcnn/std_unc_{prop}_map.tif",
            property_name=prop,
            region_name="North Algeria"
        )
    )

    results.append(
        evaluate_region(
            south_df,
             mean_tif=f"outputs_unceratinty_utcnn/mean_unc_{prop}_map.tif",
            std_tif=f"outputs_unceratinty_utcnn/std_unc_{prop}_map.tif",
            property_name=prop,
            region_name="South Algeria"
        )
    )


North Algeria points: 1964
South Algeria points: 192

📍 North Algeria — A%
Validation points: 1964
  68% → PICP=0.455, MPIW=6.866
  90% → PICP=0.607, MPIW=11.295
  95% → PICP=0.676, MPIW=13.458

📍 South Algeria — A%
Validation points: 192
  68% → PICP=0.016, MPIW=6.648
  90% → PICP=0.031, MPIW=10.936
  95% → PICP=0.042, MPIW=13.030

📍 North Algeria — L%
Validation points: 1964
  68% → PICP=0.213, MPIW=4.002
  90% → PICP=0.347, MPIW=6.583
  95% → PICP=0.422, MPIW=7.844

📍 South Algeria — L%
Validation points: 192
  68% → PICP=0.047, MPIW=4.910
  90% → PICP=0.073, MPIW=8.077
  95% → PICP=0.094, MPIW=9.624

📍 North Algeria — S%
Validation points: 1964
  68% → PICP=0.147, MPIW=5.475
  90% → PICP=0.255, MPIW=9.006
  95% → PICP=0.319, MPIW=10.731

📍 South Algeria — S%
Validation points: 192
  68% → PICP=0.042, MPIW=5.571
  90% → PICP=0.057, MPIW=9.165
  95% → PICP=0.073, MPIW=10.920

📍 North Algeria — CE
Validation points: 1964
  68% → PICP=0.122, MPIW=1.037
  90% → PICP=0.187, MPIW=1.706
  

In [36]:
final_results = pd.concat(results, ignore_index=True)

final_results.to_csv(
    "picp_mpiw_north_vs_south_algeria.csv",
    index=False
)

print("\n✅ Saved uncertainty evaluation by region")



✅ Saved uncertainty evaluation by region
